**atividade pratica:**

Esses resultados ainda não estão perfeitos; URLs que incluem "feed" não são páginas realmente visualizadas por humanos. Modifique este código para também remover URLs que incluem "/feed". Melhor ainda, extraia algumas entradas de log dessas páginas e entenda de onde essas visualizações estão vindo.

In [1]:
# Mesmo regex e mesmo arquivo de log da aula
import re

format_pat = re.compile(
    r"(?P<host>[\d\.]+)\s"
    r"(?P<identity>\S*)\s"
    r"(?P<user>\S*)\s"
    r"\[(?P<time>.*?)\]\s"
    r'"(?P<request>.*?)"\s'
    r"(?P<status>\d+)\s"
    r"(?P<bytes>\S*)\s"
    r'"(?P<referer>.*?)"\s'
    r'"(?P<user_agent>.*?)"\s*'
)

logPath = r"C:\Users\zinho\OneDrive\Documentos\Lamia\Card 11 - Prática Lidando com Dados do Mundo Real (II)\Aquivos_de_C%C3%B3digo\access_log.txt"

In [2]:
URLCounts = {}
feedEntries = []  # aqui vão as entradas completas das URLs de feed, pra analisar depois

with open(logPath, "r", encoding="utf-8", errors="ignore") as f:
    for line in (l.rstrip() for l in f):
        match = format_pat.match(line)
        if match:
            access = match.groupdict()
            request = access['request']
            fields = request.split()
            if len(fields) == 3:
                action, URL, protocol = fields
                if action == 'GET':
                    if '/feed' in URL:
                        # não conta no ranking de páginas, mas guarda a entrada inteira
                        feedEntries.append(access)
                    else:
                        URLCounts[URL] = URLCounts.get(URL, 0) + 1

results = sorted(URLCounts, key=lambda i: int(URLCounts[i]), reverse=True)

for result in results[:20]:
    print(result + ": " + str(URLCounts[result]))

/: 434
/blog/: 138
/robots.txt: 123
/sitemap_index.xml: 118
/post-sitemap.xml: 118
/page-sitemap.xml: 117
/category-sitemap.xml: 117
/orlando-headlines/: 95
/san-jose-headlines/: 85
http://51.254.206.142/httptest.php: 81
/comics-2/: 76
/travel/: 74
/entertainment/: 72
/business/: 70
/national/: 70
/national-headlines/: 70
/world/: 70
/weather/: 70
/about/: 69
/defense-sticking-head-sand/: 69


In [3]:
# quantas visualizações de feed foram tiradas do ranking
print(f'Total de acessos a URLs de feed: {len(feedEntries)}')

Total de acessos a URLs de feed: 16


In [4]:
from collections import Counter

# de onde vêm essas visualizações: referer é a página/serviço que apontou pro feed
refererCounts = Counter(e['referer'] for e in feedEntries)

print('--- Referers das páginas de feed ---')
for referer, count in refererCounts.most_common(20):
    print(referer + ": " + str(count))

--- Referers das páginas de feed ---
-: 15
http://www.nohatenews.com/?feed=rss2: 1


In [5]:
# e o user agent ajuda a ver se é gente de verdade ou leitor de feed/bot batendo ali
agentCounts = Counter(e['user_agent'] for e in feedEntries)

print('--- User agents das páginas de feed ---')
for agent, count in agentCounts.most_common(20):
    print(agent + ": " + str(count))

--- User agents das páginas de feed ---
Mozilla/5.0 (compatible; MJ12bot/v1.4.5; http://www.majestic12.co.uk/bot.php?+): 5
Mozilla/5.0 (X11; Linux i686) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/30.0.1599.66 Safari/537.36: 4
Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html): 3
-: 2
Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.0): 1
Mozilla/5.0 (compatible; AhrefsBot/5.0; +http://ahrefs.com/robot/): 1


In [6]:
# olhando algumas entradas completas de feed, pra ter o contexto todo (URL + referer + agent)
for entry in feedEntries[:20]:
    print(entry['request'] + " | referer: " + entry['referer'] + " | agent: " + entry['user_agent'])

GET /feed/ HTTP/1.1 | referer: http://www.nohatenews.com/?feed=rss2 | agent: Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.0)
GET /feed/ HTTP/1.1 | referer: - | agent: -
GET /washington-dc-sports/feed/ HTTP/1.0 | referer: - | agent: Mozilla/5.0 (compatible; MJ12bot/v1.4.5; http://www.majestic12.co.uk/bot.php?+)
GET /about/feed/ HTTP/1.0 | referer: - | agent: Mozilla/5.0 (compatible; MJ12bot/v1.4.5; http://www.majestic12.co.uk/bot.php?+)
GET /sample-page/feed/ HTTP/1.1 | referer: - | agent: Mozilla/5.0 (X11; Linux i686) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/30.0.1599.66 Safari/537.36
GET /weather/feed/ HTTP/1.1 | referer: - | agent: Mozilla/5.0 (compatible; AhrefsBot/5.0; +http://ahrefs.com/robot/)
GET /san-francisco-sports/feed/ HTTP/1.1 | referer: - | agent: Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)
GET /feeds/tampa-bay-times-top-news/ HTTP/1.1 | referer: - | agent: Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)
GET /f